In [ ]:
import h5py
import numpy as np
import torch
from datasets import load_dataset
from evo import Evo
from tqdm import tqdm

# --- CONFIGURATION ---
MODEL_NAME = "evo-1-8k-base"  # Options: evo-1-8k-base, evo-1-131k-base
TASK_NAME = "dna_rna"
OUTPUT_H5 = f"{TASK_NAME}_Evo-1-8k.h5"

MAX_LEN = 4096
STRIDE = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 1. DATASET ---
dataset = load_dataset("pabloarozarenad/vGUE-benchmark", TASK_NAME, split="test")
sequences = dataset["sequence"]
labels = [str(lbl) for lbl in dataset["label"]]
label_names = sorted(list(set(labels)))
label_to_id = {name: idx for idx, name in enumerate(label_names)}
label_ids = np.array([label_to_id[lbl] for lbl in labels], dtype=np.int64)

# --- 2. LOAD EVO MODEL ---
evo_wrapper = Evo(MODEL_NAME)
model = evo_wrapper.model.to(DEVICE)
tokenizer = evo_wrapper.tokenizer
model.eval()

# Helper function to generate overlapping token windows
def get_overlapping_windows(token_ids, max_len=4096, stride=512):
    step = max_len - stride
    windows = []
    start = 0
    while start < len(token_ids):
        windows.append(token_ids[start : start + max_len])
        if start + max_len >= len(token_ids):
            break
        start += step
    return windows

# --- 3. EXTRACTION ---
all_embeddings = []

for sequence in tqdm(sequences, desc="Extracting Evo Embeddings"):
    token_ids = tokenizer.tokenize(sequence)
    if torch.is_tensor(token_ids):
        token_ids = token_ids.detach().cpu().tolist()
    
    windows = get_overlapping_windows(token_ids, max_len=MAX_LEN, stride=STRIDE)
    window_embs = []
    
    for win in windows:
        input_ids = torch.tensor(win, dtype=torch.long, device=DEVICE).unsqueeze(0)
        
        with torch.inference_mode():
            logits = model(input_ids)  # Shape: [1, win_len, 512]
            if isinstance(logits, tuple):
                logits = logits[0]
                
        # Unmasked token-level max pooling over native 512-char vocabulary
        win_emb = torch.max(logits, dim=1).values  # Shape: [1, 512]
        window_embs.append(win_emb.float().cpu())
        
    # Window-level mean aggregation
    seq_emb = torch.mean(torch.cat(window_embs, dim=0), dim=0)
    all_embeddings.append(seq_emb.numpy())

final_matrix = np.vstack(all_embeddings)

# --- 4. SAVE ---
string_dtype = h5py.string_dtype(encoding="utf-8")
with h5py.File(OUTPUT_H5, "w") as f:
    f.create_dataset("embeddings", data=final_matrix)
    f.create_dataset("labels", data=np.array(labels, dtype=object), dtype=string_dtype)
    f.create_dataset("label_ids", data=label_ids)
    f.create_dataset("label_names", data=np.array(label_names, dtype=object), dtype=string_dtype)
    f.attrs["model_name"] = MODEL_NAME
    f.attrs["representation_type"] = "max_pooled_causal_lm_logits"
    f.attrs["embedding_dimension"] = 512

print(f"Saved {final_matrix.shape} matrix to {OUTPUT_H5}")